# Customer Catalogue GP Homologation

## Step 1 - Load, validate and normalize Global Parents

This notebook:

- Loads the customer catalogue
- Validates required columns
- Counts unique Ship To Numbers per Global Parent
- Creates a normalized GP name
- Exports a summary for review

The original catalogue is NEVER modified.

In [27]:
import pandas as pd
import re
import unicodedata
from pathlib import Path

In [28]:
# ==========================================
# Configuration
# ==========================================

INPUT_FILE = r'C:\Users\Ivan.DeLaFuente\OneDrive - Quaker Houghton\Documents\QH WorkDay\July 2026\Industrial Market\Industrial_Market.csv'

OUTPUT_FOLDER = "output"

GP_COLUMN = "Global Parent"

SHIP_TO_COLUMN = "Ship To Number"

REMOVE_ACCENTS = True
REMOVE_PUNCTUATION = True
REMOVE_EXTRA_SPACES = True
REMOVE_LEGAL_SUFFIXES = True

In [29]:
LEGAL_SUFFIXES = [

    "INCORPORATED",
    "INC",
    "CORPORATION",
    "CORP",
    "COMPANY",
    "CO",
    "LIMITED",
    "LTD",
    "LLC",
    "LLP",
    "PLC",
    "LP",
    "GMBH",
    "AG",
    "BV",
    "NV",
    "SA",
    "SAS",
    "SPA",
    "SRL",
    "PTY",
    "PVT",
    "KK"

]

In [30]:
def normalize_gp(name):

    if pd.isna(name):
        return ""

    text = str(name).upper()

    if REMOVE_ACCENTS:
        text = unicodedata.normalize("NFKD", text)
        text = "".join(c for c in text if not unicodedata.combining(c))

    if REMOVE_PUNCTUATION:
        text = re.sub(r"[^\w\s]", " ", text)

    if REMOVE_LEGAL_SUFFIXES:

        words = text.split()

        words = [
            word
            for word in words
            if word not in LEGAL_SUFFIXES
        ]

        text = " ".join(words)

    if REMOVE_EXTRA_SPACES:
        text = re.sub(r"\s+", " ", text).strip()

    return text

In [31]:
df = pd.read_csv(INPUT_FILE)

print(f"Rows loaded: {len(df):,}")

Rows loaded: 49,493


In [32]:
required_columns = [

    GP_COLUMN,
    SHIP_TO_COLUMN

]

missing = [

    column
    for column in required_columns
    if column not in df.columns

]

if missing:

    raise ValueError(
        f"Missing columns: {missing}"
    )

print("All required columns found.")

All required columns found.


In [33]:
df["Normalized GP"] = df[GP_COLUMN].apply(normalize_gp)

df["Normalization Changed"] = (
    df[GP_COLUMN].str.upper().fillna("")
    !=
    df["Normalized GP"]
)

In [34]:
gp_summary = (

    df.groupby(GP_COLUMN)
      .agg(

          Ship_To_Count=(
              SHIP_TO_COLUMN,
              "nunique"
          ),

          Row_Count=(
              SHIP_TO_COLUMN,
              "count"
          ),

          GP_Country=(
              "GP Country",
              "first"
          ),

          Normalized_GP=(
              "Normalized GP",
              "first"
          ),

          Normalization_Changed=(
              "Normalization Changed",
              "first")

      )

      .reset_index()

)

In [35]:
gp_summary = gp_summary.sort_values(

    by=[
        "Ship_To_Count",
        GP_COLUMN
    ],

    ascending=[
        False,
        True
    ]

)

In [36]:
gp_summary.head(20)

,Global Parent,Ship_To_Count,Row_Count,GP_Country,Normalized_GP,Normalization_Changed
16887,TOTAL,121,272,France,TOTAL,False
14566,SAMPLE ACCOUNT,105,185,NaN,SAMPLE ACCOUNT,False
316,ADJUSTMENT,94,202,England,ADJUSTMENT,False
5561,FASTENAL,94,178,United States Of America,FASTENAL,False
12565,PARKER HANNIFIN,87,151,United States Of America,PARKER HANNIFIN,False
9978,Linde,76,145,Ireland,LINDE,False
2078,BODYCOTE,70,131,England,BODYCOTE,False
4693,EATON,66,133,United States Of America,EATON,False
12693,PENINSULAR DE LUBRICANTES,65,70,Spain,PENINSULAR DE LUBRICANTES,False
3967,DANFOSS,50,100,Denmark,DANFOSS,False


In [37]:
Path(OUTPUT_FOLDER).mkdir(exist_ok=True)

output_file = Path(OUTPUT_FOLDER) / "01_gp_summary.csv"

gp_summary.to_csv(

    output_file,

    index=False

)

print(f"Saved to: {output_file}")

Saved to: output\01_gp_summary.csv


In [39]:
exact_groups = (
    gp_summary
    .groupby("Normalized_GP")
    .agg(
        Variants=(GP_COLUMN, list),
        Number_of_Variants=(GP_COLUMN, "nunique"),
        Total_Ship_To_Count=("Ship_To_Count", "sum"),
        Total_Row_Count=("Row_Count", "sum"),
        Countries=("GP_Country", lambda x: sorted(set(x.dropna()))),
        Any_Normalization_Changed=("Normalization_Changed", "any")
    )
    .reset_index()
)

In [40]:
exact_groups = exact_groups[
    exact_groups["Number_of_Variants"] > 1
].copy()

In [41]:
exact_groups = exact_groups.sort_values(
    by=[
        "Total_Ship_To_Count",
        "Number_of_Variants"
    ],
    ascending=False
)

In [42]:
exact_groups.head(20)

,Normalized_GP,Variants,Number_of_Variants,Total_Ship_To_Count,Total_Row_Count,Countries,Any_Normalization_Changed
6170,GENUINE PARTS,"[GENUINE PARTS, GENUINE PARTS COMPANY]",2,20,43,[United States Of America],True
12361,PENTAIR,"[PENTAIR, Pentair]",2,17,27,[United States Of America],False
10910,MOLEX,"[MOLEX INC, MOLEX]",2,16,27,[United States Of America],True
16063,TENNECO,"[TENNECO, Tenneco]",2,12,20,[United States Of America],False
635,ALFA LAVAL,"[ALFA LAVAL, ALFA LAVAL INC]",2,11,21,[Sweden],True
8450,JOHNSON CONTROLS,"[JOHNSON CONTROLS INC, Johnson Controls]",2,10,23,[United States Of America],True
8634,KAMAN,"[Kaman Corporation, KAMAN]",2,10,19,[United States Of America],True
14284,SCHAEFFLER,"[SCHAEFFLER, Schaeffler]",2,10,16,[Germany],False
3415,CONDO,"[CONDO, CONDO INC, Condo Inc]",3,9,11,[United States Of America],True
10281,MECALUX,"[MECALUX, MECALUX SA]",2,9,18,[Spain],True


In [43]:
output_file = Path(OUTPUT_FOLDER) / "02_exact_normalized_groups.csv"

exact_groups.to_csv(
    output_file,
    index=False
)

print(f"Saved to: {output_file}")

Saved to: output\02_exact_normalized_groups.csv


In [ ]:
entities = exact_groups.copy()

entities = entities.rename(
    columns={
        "Normalized_GP": "Representative",
        "Variants": "Members",
        "Total_Ship_To_Count": "Ship_To_Count",
        "Total_Row_Count": "Row_Count",
        "Countries": "Countries"
    }
)

In [ ]:
entities.insert(
    0,
    "Entity_ID",
    [
        f"E{i:06d}"
        for i in range(1, len(entities) + 1)
    ]
)

In [ ]:
grouped_gps = set()

for members in entities["Members"]:
    grouped_gps.update(members)

singletons = gp_summary[
    ~gp_summary[GP_COLUMN].isin(grouped_gps)
].copy()

In [ ]:
singletons["Representative"] = singletons["Normalized_GP"]

singletons["Members"] = singletons[GP_COLUMN]

singletons["Countries"] = singletons["GP_Country"]


start = len(entities) + 1

singletons.insert(
    0,
    "Entity_ID",
    [
        f"E{i:06d}"
        for i in range(start, start + len(singletons))
    ]
)

In [ ]:
singletons = singletons[
    [
        "Entity_ID",
        "Representative",
        "Members",
        "Ship_To_Count",
        "Row_Count",
        "Countries"
    ]
]

In [ ]:
entities = pd.concat(
    [
        entities,
        singletons
    ],
    ignore_index=True
)

In [ ]:
entities = entities.sort_values(
    by="Ship_To_Count",
    ascending=False
)

In [ ]:
entities.head(20)

In [ ]:
def list_to_string(value):

    if isinstance(value, list):
        return " | ".join(value)

    if pd.isna(value):
        return ""

    return str(value)

entities["Members"] = entities["Members"].apply(list_to_string)

entities["Countries"] = entities["Countries"].apply(list_to_string)


output_file = Path(OUTPUT_FOLDER) / "03_entities.csv"
entities.to_csv(
    output_file,
    index=False
)

print(f"Saved to: {output_file}")

NameError: name 'entities' is not defined

Temporal cells

In [44]:
print(len(exact_groups))

561


In [45]:
exact_groups.head(20)

,Normalized_GP,Variants,Number_of_Variants,Total_Ship_To_Count,Total_Row_Count,Countries,Any_Normalization_Changed
6170,GENUINE PARTS,"[GENUINE PARTS, GENUINE PARTS COMPANY]",2,20,43,[United States Of America],True
12361,PENTAIR,"[PENTAIR, Pentair]",2,17,27,[United States Of America],False
10910,MOLEX,"[MOLEX INC, MOLEX]",2,16,27,[United States Of America],True
16063,TENNECO,"[TENNECO, Tenneco]",2,12,20,[United States Of America],False
635,ALFA LAVAL,"[ALFA LAVAL, ALFA LAVAL INC]",2,11,21,[Sweden],True
8450,JOHNSON CONTROLS,"[JOHNSON CONTROLS INC, Johnson Controls]",2,10,23,[United States Of America],True
8634,KAMAN,"[Kaman Corporation, KAMAN]",2,10,19,[United States Of America],True
14284,SCHAEFFLER,"[SCHAEFFLER, Schaeffler]",2,10,16,[Germany],False
3415,CONDO,"[CONDO, CONDO INC, Condo Inc]",3,9,11,[United States Of America],True
10281,MECALUX,"[MECALUX, MECALUX SA]",2,9,18,[Spain],True


In [46]:
gp_summary['Normalization_Changed'].value_counts()

Normalization_Changed
False    15700
True      3080
Name: count, dtype: int64

In [ ]:
print(len(entities))